# 00 · RUN ME: one-click runner for the TA-LSTMAE experiments

This notebook works in **VS Code, Jupyter or Google Colab**. It performs all the setup steps in order:
1. finds the project folder (and mounts Google Drive on Colab);
2. installs any missing packages and runs the unit tests;
3. checks the hardware and the dataset files (CICIDS-2018 can be downloaded automatically);
4. starts the Elasticsearch/Logstash testbed with Docker when notebook 03 is selected (local machine only);
5. executes notebooks **01 → 02 → 03** and shows live progress;
6. displays the key result tables and packs them into `results_to_send.zip`.

**How to use:** edit the *SETTINGS* cell only, then choose **Run All**.
Recommended first run: `MODE = "smoke"`. It takes about 15–30 minutes and only checks that everything works; its numbers are meaningless. After that, switch to `"quick"` (3 seeds) or `"full"` (5 seeds, as reported in the paper).

Interrupted runs continue where they stopped (`RESUME = True`). Finished models and seeds are skipped automatically, so on a laptop or Colab you can safely stop and run again.

## SETTINGS (edit this cell only)

In [ ]:
MODE = "smoke"              # "smoke" = code check, synthetic data (INVALID numbers) | "quick" = 3 seeds | "full" = 5 seeds (paper)
DATASETS = ["nsl_kdd", "unsw_nb15", "cicids2018"]   # remove entries to run one dataset per session
RUN = ["01", "02", "03"]    # notebooks to execute, in this order (03 needs the checkpoints written by 01)
RESUME = True               # skip models/seeds that are already finished

COLAB_PROJECT_DIR = "/content/drive/MyDrive/ta-lstmae"   # used only on Google Colab
AUTO_DOWNLOAD_CICIDS = True                               # download CICIDS-2018 (02-14-2018) from the public AWS bucket if missing

# Notebook 03 (end-to-end latency)
START_ELK_WITH_DOCKER = True        # try "docker compose up -d elasticsearch logstash" (local machine with Docker only)
ES_URL = "http://localhost:9200"
LOGSTASH_URL = "http://localhost:8080"
SOAR_WEBHOOK_URL = ""               # optional; empty = blocklist-file response action
LATENCY_EVENTS = 5000               # events replayed per pipeline; None = complete CICIDS-2018 test stream

## 1. Project folder

In [ ]:
import os, sys, json, time, subprocess, shutil, threading, importlib, urllib.request
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path(COLAB_PROJECT_DIR)
else:
    ROOT = Path.cwd() if (Path.cwd() / "talstmae").exists() else Path.cwd().parent
assert (ROOT / "talstmae").exists(), f"Project folder not found at {ROOT}. Set COLAB_PROJECT_DIR or open this notebook from ta-lstmae/notebooks."
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
assert MODE in ("smoke", "quick", "full"), "MODE must be 'smoke', 'quick' or 'full'"
OUT = ROOT / ("outputs_SMOKE_TEST_INVALID" if MODE == "smoke" else "outputs")
OUT.mkdir(exist_ok=True)
print("Project folder:", ROOT, "| environment:", "Google Colab" if IN_COLAB else "local (VS Code / Jupyter)",
      "| Python", sys.version.split()[0], "|", sys.executable)

## 2. Packages (only missing ones are installed) and unit tests

In [ ]:
NEEDED = {"tensorflow": "tensorflow>=2.16", "keras": "keras>=3.0", "numpy": "numpy", "pandas": "pandas",
          "sklearn": "scikit-learn", "scipy": "scipy", "tabulate": "tabulate", "nbformat": "nbformat",
          "nbclient": "nbclient", "ipykernel": "ipykernel", "pytest": "pytest", "psutil": "psutil"}
missing = [pkg for mod, pkg in NEEDED.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("Installing:", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
    importlib.invalidate_caches()
else:
    print("All required packages are installed.")
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True,
                   env={**os.environ, "TF_CPP_MIN_LOG_LEVEL": "3"})
print(r.stdout.strip().splitlines()[-1] if r.stdout.strip() else r.stderr[-2000:])
assert r.returncode == 0, "Unit tests failed - send this output to the authors' assistant:\n" + r.stdout[-3000:] + r.stderr[-2000:]

## 3. Hardware check

In [ ]:
import psutil
import tensorflow as tf
gpus = tf.config.list_physical_devices("GPU")
ram = psutil.virtual_memory().total / 2**30
disk = shutil.disk_usage(ROOT).free / 2**30
print(f"TensorFlow {tf.__version__} | GPU: {[g.name for g in gpus] or 'none (CPU only)'} | RAM {ram:.1f} GB | "
      f"CPU cores {os.cpu_count()} | free disk {disk:.1f} GB")
if ram < 12: print("⚠ Less than 12 GB RAM: close other applications; CICIDS-2018 loading is memory-intensive.")
if disk < 5: print("⚠ Less than 5 GB free disk space.")
if not gpus and MODE != "smoke":
    print("⚠ No GPU: 'quick'/'full' runs can take one or more days on a CPU. Consider Google Colab (GPU) for 01 and 02,\n"
          "  or run one dataset per session (edit DATASETS). Interrupted runs resume automatically.")

## 4. Dataset files

In [ ]:
FILES = {"nsl_kdd": ["nsl-kdd/KDDTrain+.txt", "nsl-kdd/KDDTest+.txt"],
         "unsw_nb15": ["unsw-nb15/UNSW_NB15_training-set.csv", "unsw-nb15/UNSW_NB15_testing-set.csv"],
         "cicids2018": ["cicids2018/02-14-2018.csv"]}
HOWTO = {"nsl_kdd": "NSL-KDD: Canadian Institute for Cybersecurity (UNB) dataset page -> KDDTrain+.txt, KDDTest+.txt",
         "unsw_nb15": "UNSW-NB15: UNSW Canberra dataset page -> 'Training and Testing Sets' -> UNSW_NB15_training-set.csv, UNSW_NB15_testing-set.csv",
         "cicids2018": "CSE-CIC-IDS2018 (AWS Open Data): 'Processed Traffic Data for ML Algorithms/Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv' -> rename to 02-14-2018.csv"}
CIC_URL = ("https://cse-cic-ids2018.s3.ca-central-1.amazonaws.com/Processed%20Traffic%20Data%20for%20ML%20Algorithms/"
           "Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv")

def download(url, dest):
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_suffix(".part")
    with urllib.request.urlopen(url, timeout=60) as r, open(tmp, "wb") as f:
        total, done, t0 = int(r.headers.get("Content-Length", 0)), 0, time.time()
        while chunk := r.read(1 << 20):
            f.write(chunk); done += len(chunk)
            if time.time() - t0 > 5:
                print(f"  {done / 2**20:,.0f} / {total / 2**20:,.0f} MB", end="\r"); t0 = time.time()
    tmp.rename(dest)

if MODE == "smoke":
    print("MODE = 'smoke': datasets are not needed (synthetic placeholder data is used).")
else:
    cic = ROOT / "data" / FILES["cicids2018"][0]
    alt = sorted((ROOT / "data" / "cicids2018").glob("*02-14-2018*.csv")) if (ROOT / "data" / "cicids2018").exists() else []
    if alt and not cic.exists():
        cic = alt[0]; FILES["cicids2018"] = [str(cic.relative_to(ROOT / "data")).replace("\\", "/")]
        print("Using CICIDS-2018 file:", cic.name)
    if "cicids2018" in DATASETS and not cic.exists() and AUTO_DOWNLOAD_CICIDS:
        print("Downloading CICIDS-2018 (02-14-2018, ~350 MB) ...")
        try:
            download(CIC_URL, cic); print("\n  done:", cic)
        except Exception as e:
            print("\n  automatic download failed:", e, "\n  ->", HOWTO["cicids2018"])
    ok = True
    for ds in DATASETS:
        for rel in FILES[ds]:
            p = ROOT / "data" / rel
            status = f"OK ({p.stat().st_size / 2**20:,.1f} MB)" if p.exists() else "MISSING"
            ok &= p.exists()
            print(f"  {'✓' if p.exists() else '✗'} data/{rel:<45s} {status}")
        if not all((ROOT / "data" / rel).exists() for rel in FILES[ds]):
            print("    how to obtain:", HOWTO[ds])
    assert ok, "Some dataset files are missing (see above). Place them in ta-lstmae/data/ and run this cell again."

## 5. Write run settings and prepare the SIEM testbed (notebook 03)

In [ ]:
SEEDS = {"smoke": [42, 43], "quick": [42, 43, 44], "full": [42, 43, 44]}[MODE]
settings = {"smoke_test": MODE == "smoke", "resume": RESUME, "latency_events": LATENCY_EVENTS,
            "config": {"datasets": DATASETS, "seeds": SEEDS}}
SETTINGS_FILE = ROOT / "run_settings.json"
SETTINGS_FILE.write_text(json.dumps(settings, indent=2))
os.environ["TALSTMAE_SETTINGS"] = str(SETTINGS_FILE)        # read by notebooks 01-03
print("run settings:", settings)

def es_alive(url):
    try:
        with urllib.request.urlopen(url, timeout=3) as r:
            return r.status == 200
    except Exception:
        return False

for k in ("ES_URL", "LOGSTASH_URL", "SOAR_WEBHOOK_URL"):
    os.environ.pop(k, None)
if "03" in RUN:
    if not es_alive(ES_URL) and START_ELK_WITH_DOCKER and not IN_COLAB:
        if not shutil.which("docker"):
            print("⚠ Docker is not installed: install Docker Desktop to measure SIEM ingestion.")
        elif subprocess.run(["docker", "info"], capture_output=True).returncode != 0:
            print("⚠ Docker is installed but not running: start Docker Desktop, then run this cell again.")
        else:
            print("Starting Elasticsearch + Logstash with Docker (first start downloads ~1.5 GB) ...")
            subprocess.run(["docker", "compose", "up", "-d", "elasticsearch", "logstash"], cwd=ROOT / "docker")
            for _ in range(60):
                if es_alive(ES_URL): break
                time.sleep(5)
    if es_alive(ES_URL):
        os.environ["ES_URL"] = ES_URL
        print("✓ Elasticsearch reachable at", ES_URL)
        ls_ok = False
        for _ in range(24):                                   # Logstash starts more slowly than Elasticsearch
            try:
                req = urllib.request.Request(LOGSTASH_URL, data=b'{"event": {"id": "warmup"}}',
                                             headers={"Content-Type": "application/json"})
                ls_ok = urllib.request.urlopen(req, timeout=3).status in (200, 201)
            except Exception:
                ls_ok = False
            if ls_ok: break
            time.sleep(5)
        if ls_ok:
            os.environ["LOGSTASH_URL"] = LOGSTASH_URL; print("✓ Logstash reachable at", LOGSTASH_URL)
        else:
            print("⚠ Logstash not reachable: only direct Elasticsearch ingestion will be measured.")
    else:
        print("⚠ Elasticsearch not reachable: notebook 03 will measure model + JSON + file latency only "
              "(SIEM ingestion NOT measured)." + (" Docker is not available on Colab; run 03 on a local machine." if IN_COLAB else ""))
    if SOAR_WEBHOOK_URL:
        os.environ["SOAR_WEBHOOK_URL"] = SOAR_WEBHOOK_URL

## 6. Execute the notebooks (live progress below; full log in `outputs/progress.log`)

In [ ]:
import nbformat
from nbclient import NotebookClient
from nbclient.exceptions import CellExecutionError

NOTEBOOKS = {"01": "01_reproduce_main_results.ipynb", "02": "02_head_to_head_baselines.ipynb",
             "03": "03_end_to_end_latency.ipynb"}
EXEC_DIR = OUT / "executed_notebooks"; EXEC_DIR.mkdir(parents=True, exist_ok=True)
LOG = OUT / "progress.log"

def tail_log(stop, start_size):
    pos = start_size
    while not stop.is_set():
        if LOG.exists() and LOG.stat().st_size > pos:
            with open(LOG) as f:
                f.seek(pos); new = f.read(); pos = f.tell()
            for line in new.splitlines():
                print("   ", line, flush=True)
        stop.wait(5)

def run_notebook(key):
    name = NOTEBOOKS[key]
    nb = nbformat.read(ROOT / "notebooks" / name, as_version=4)
    code_cells = [i for i, c in enumerate(nb.cells) if c.cell_type == "code"]
    t_start = {}
    def on_start(cell, cell_index, **kw):
        if cell.cell_type != "code":
            return
        t_start[cell_index] = time.time()
        first = next((l for l in cell.source.splitlines() if l.strip() and not l.startswith(("import", "from", "#"))), "")
        print(f"  ▶ [{code_cells.index(cell_index) + 1}/{len(code_cells)}] {first[:90]}", flush=True)
    def on_complete(cell, cell_index, **kw):
        if cell_index not in t_start:
            return
        print(f"    ✓ {time.time() - t_start[cell_index]:.0f}s", flush=True)
    client = NotebookClient(nb, timeout=None, kernel_name="python3", allow_errors=False,
                            resources={"metadata": {"path": str(ROOT / "notebooks")}},
                            on_cell_start=on_start, on_cell_executed=on_complete)
    stop = threading.Event()
    th = threading.Thread(target=tail_log, args=(stop, LOG.stat().st_size if LOG.exists() else 0), daemon=True)
    th.start(); t0 = time.time()
    print(f"\n=== {name} ===", flush=True)
    try:
        client.execute()
        print(f"=== {name} finished in {(time.time() - t0) / 60:.1f} min ===", flush=True)
        return True
    except CellExecutionError as e:
        msg = str(e)
        print(f"\n✗ {name} stopped with an error after {(time.time() - t0) / 60:.1f} min:\n", msg[-2500:])
        print("Fix the problem (or send this message), then Run All again: finished parts are resumed.")
        return False
    finally:
        stop.set(); th.join(timeout=10)
        nbformat.write(nb, EXEC_DIR / name)

for key in RUN:
    if not run_notebook(key):
        break

## 7. Key results and the package to send

In [ ]:
import pandas as pd
from IPython.display import display, Markdown
if MODE == "smoke":
    display(Markdown("**SMOKE TEST: the numbers below are meaningless; they only show that every step ran.**"))
show = [("Comparison with the manuscript (seed 42)", "tables/comparison_with_manuscript.csv"),
        ("Table 6: temporal metrics (causal MTTD primary)", "tables/table6_temporal_metrics_causal_primary.csv"),
        ("Table 4: TA-LSTMAE mean ± SD over seeds", "tables/table4_ta_lstmae_mean_sd.csv"),
        ("Head-to-head comparison (mean ± SD)", "tables_head_to_head.csv"),
        ("Head-to-head paired tests", "head_to_head_paired_tests.csv"),
        ("End-to-end latency (ms)", "latency/latency_summary.csv")]
for title, rel in show:
    p = OUT / rel
    if p.exists():
        display(Markdown(f"### {title}")); display(pd.read_csv(p).round(4).head(60))
import zipfile
zip_path = ROOT / ("results_SMOKE_TEST_INVALID.zip" if MODE == "smoke" else "results_to_send.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file() and p.suffix not in (".keras",):
            z.write(p, p.relative_to(ROOT))
print(f"Results packed in {zip_path} ({zip_path.stat().st_size / 2**20:.1f} MB) - send this file for the manuscript revision.")